In [8]:
import pandas as pd
import json


In [9]:
df = pd.read_csv('milestone2_evaluationshruti_output.csv')
df.head(5)


,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag,expected_apr,expected_term,expeted_payment,expected_penalty,apr_score,term_score,penalty_score,total_score,quality_percent
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low,10.49,24,959,NaN,1,1,0,2,66.666667
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low,3.78,24,804,Early termination fee $300,1,1,1,3,100.000000
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High,5.41,24,774,Late fee $50,1,1,1,3,100.000000
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High,5.26,48,1028,Late fee $25,1,1,1,3,100.000000
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low,5.97,36,1180,NaN,1,1,0,2,66.666667


In [1]:
expected_output_format = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}


In [2]:
PROMPT = """
You are an information extraction system.

Extract ONLY the following fields from the contract text:
- APR
- Term (in months)
- Monthly payment
- Penalty clause

Rules:
- Return output strictly in JSON format
- If a value is not mentioned, return null
- Do NOT infer or assume values

Return JSON with keys:
apr, term_months, monthly_payment, penalty_clause
"""

In [3]:
def call_llm(prompt_text):
    """
    Dummy LLM response for baseline testing.
    """
    return json.dumps({
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    })

In [7]:
def extract_fields(contract_text):
    prompt_filled = PROMPT.format(text=contract_text)
    llm_response = call_llm(prompt_filled)
    return json.loads(llm_response)


In [8]:
sample_contracts = df.sample(5, random_state=1)
sample_contracts


NameError: name 'df' is not defined

In [5]:
extracted_rows = []

for _, row in sample_contracts.iterrows():
    extracted = extract_fields(row["contract_text"])

    extracted_rows.append({
        "contract_text": row["contract_text"],

        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"],

        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_payment"],
        "expected_penalty": row["expected_penalty"]
    })

comparison_df = pd.DataFrame(extracted_rows)
comparison_df


NameError: name 'sample_contracts' is not defined

In [ ]:
def check_match(predicted, actual):
    if pd.isna(predicted) and pd.isna(actual):
        return 1
    if predicted == actual:
        return 1
    return 0


In [ ]:
comparison_df["apr_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_apr"], r["expected_apr"]), axis=1
)

comparison_df["term_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_term"], r["expected_term"]), axis=1
)

comparison_df["payment_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_payment"], r["expected_payment"]), axis=1
)

comparison_df["penalty_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_penalty"], r["expected_penalty"]), axis=1
)

comparison_df


,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty,apr_match,term_match,payment_match,penalty_match
0,Monthly payment $315 with late fee $30 if dela...,None,None,None,None,NaN,NaN,315,Late fee $30,1,1,0,0
1,This contract has a tenure of 72 months and AP...,None,None,None,None,9.75,72.0,NaN,NaN,0,0,1,1
2,APR 5.9% with no penalties mentioned.,None,None,None,None,5.90,NaN,NaN,NaN,0,1,1,1
3,Customer agrees to a 24 month contract with AP...,None,None,None,None,6.80,24.0,NaN,NaN,0,0,1,1
4,APR 14.6% applies with early termination charg...,None,None,None,None,14.60,NaN,NaN,Early termination fee $500,0,1,1,0
